In [ ]:
!mkdir ImageNet
%cd ImageNet
!wget https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar
!wget https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz
%cd ..

In [ ]:
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import numpy as np

activation = {}


def get_activation(name):
    def hook(model, input, output):
        activation[name] = output.detach()

    return hook


def get_correlation(batch_size, layers1, layers2):
    layers1 = layers1[:, 0].reshape(batch_size, -1)
    layers2 = layers2[:, 0].reshape(batch_size, -1)
    correlations = np.zeros_like(layers1.cpu().detach().numpy()[0])
    for i in range(layers1.shape[1]):
        neuron_i = layers1[:, i]
        max_corr = 0
        for j in range(layers2.shape[1]):
            neuron_j = layers2[:, j]
            stacked = torch.stack((neuron_i, neuron_j))
            corr = torch.corrcoef(stacked)[0, 1]
            corr = np.abs(corr.cpu().detach().numpy())
            if corr is np.nan:
                corr = 0
            if corr > max_corr:
                max_corr = corr
        correlations[i] = max_corr

    return correlations.mean()


def mean_correlation(batch_size, layers1, layers2):
    return (get_correlation(batch_size, layers1, layers2) + get_correlation(batch_size, layers2, layers1)) / 2


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)
    print(f"timm version: {timm.__version__}")

    # Check available EfficientNet models
    print("\n" + "=" * 70)
    print("Checking available EfficientNet models...")
    print("=" * 70)

    available_efficientnet = timm.list_models('efficientnet*', pretrained=True)
    print(f"\nTotal EfficientNet models available: {len(available_efficientnet)}")

    if len(available_efficientnet) > 0:
        print("\nAvailable EfficientNet models (first 20):")
        for model in sorted(available_efficientnet)[:20]:
            print(f"  - {model}")
        if len(available_efficientnet) > 20:
            print(f"  ... and {len(available_efficientnet) - 20} more")

    # EfficientNet V1 and V2 models to compare
    print("\n" + "=" * 70)
    print("Loading EfficientNet models for comparison...")
    print("=" * 70)

    # Define model candidates
    # Format: (model_id, display_name, num_stages)
    model_candidates = [
        # EfficientNet V1 (original)
        ("efficientnet_b0", "EfficientNet-B0", 7),
        ("efficientnet_b1", "EfficientNet-B1", 7),
        ("efficientnet_b2", "EfficientNet-B2", 7),
        ("efficientnet_b3", "EfficientNet-B3", 7),
        ("efficientnet_b4", "EfficientNet-B4", 7),
        ("efficientnet_b5", "EfficientNet-B5", 7),

        # EfficientNet V2 (improved)
        ("efficientnetv2_s", "EfficientNetV2-S", None),
        ("efficientnetv2_m", "EfficientNetV2-M", None),
        ("efficientnetv2_l", "EfficientNetV2-L", None),
    ]

    # Load models
    efficientnet_models = []
    model_names = []
    model_info = []

    for model_id, name, num_stages in model_candidates:
        if model_id not in available_efficientnet:
            print(f"\n  ✗ {name} ({model_id}) not available, skipping...")
            continue

        print(f"\n  Attempting to load: {name} ({model_id})...")
        try:
            model = timm.create_model(model_id, pretrained=True).to(device)
            model.eval()

            # Get model structure info
            if hasattr(model, 'blocks'):
                actual_stages = len(model.blocks)
            elif hasattr(model, 'stages'):
                actual_stages = len(model.stages)
            else:
                # Try to count blocks
                actual_stages = num_stages if num_stages else 7

            print(f"    ✓ Loaded successfully!")
            print(f"      Stages: {actual_stages}")

            efficientnet_models.append(model)
            model_names.append(name)
            model_info.append({
                'stages': actual_stages,
                'model_id': model_id
            })

        except Exception as e:
            print(f"    ✗ Failed: {str(e)[:100]}")

    if len(efficientnet_models) < 2:
        print("\n" + "=" * 70)
        print("ERROR: Need at least 2 models for comparison.")
        print("=" * 70)
        print("\nTrying to load at least B0 and B1...")

        # Fallback: just load B0 and B1
        minimal_models = [
            ("efficientnet_b0", "EfficientNet-B0", 7),
            ("efficientnet_b1", "EfficientNet-B1", 7),
        ]

        for model_id, name, num_stages in minimal_models:
            try:
                model = timm.create_model(model_id, pretrained=True).to(device)
                model.eval()
                efficientnet_models.append(model)
                model_names.append(name)
                model_info.append({'stages': 7, 'model_id': model_id})
                print(f"  ✓ Loaded {name}")
            except:
                pass

        if len(efficientnet_models) < 2:
            print("Still failed. Please check your timm installation.")
            exit(1)

    print("\n" + "=" * 70)
    print(f"Successfully loaded {len(efficientnet_models)} models!")
    print("=" * 70)

    # Create standard ImageNet transform
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        normalize,
    ])

    # Load ImageNet dataset
    print("\nLoading ImageNet dataset...")
    imagenet_dataset = datasets.ImageNet('/data/ImageNet', split='val', transform=transform)
    data_loader = DataLoader(imagenet_dataset, batch_size=10, shuffle=True, num_workers=1, pin_memory=True)

    print(f"\nUsing method: {method.upper()}")
    print("\nModel specifications:")
    print("-" * 70)
    print(f"{'Model':<20} {'Stages':<10} {'Hook Layer':<15}")
    print("-" * 70)

    # Determine which layer/stage to hook for each model
    hook_targets = []
    for i, name in enumerate(model_names):
        stages = model_info[i]['stages']
        middle_stage = stages // 2  # Middle stage
        hook_targets.append(middle_stage)
        print(f"{name:<20} {stages:<10} Stage {middle_stage:<15}")

    print("\n" + "=" * 70)
    print("Starting comparison...")
    print("=" * 70)

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(data_loader):
            images = images.to(device)
            labels = labels.to(device)
            batch_size = images.shape[0]
            activations = []

            print(f"\nBatch {batch_idx + 1}, size: {batch_size}")
            print("\nExtracting activations from middle stages:")

            for i, model in enumerate(efficientnet_models):
                hook_name = f"stage_{i}"

                try:
                    # EfficientNet structure: model.blocks[stage_idx]
                    if hasattr(model, 'blocks'):
                        stage_idx = hook_targets[i]
                        model.blocks[stage_idx].register_forward_hook(
                            get_activation(hook_name)
                        )
                    elif hasattr(model, 'stages'):
                        stage_idx = hook_targets[i]
                        model.stages[stage_idx].register_forward_hook(
                            get_activation(hook_name)
                        )
                    else:
                        print(f"  ✗ Cannot find blocks/stages in {model_names[i]}")
                        continue

                    # Forward pass
                    outputs = model(images)

                    if hook_name in activation:
                        act = activation[hook_name].detach()
                        activations.append(act)

                        # Show shape and channel info
                        if len(act.shape) == 4:
                            print(f"  {model_names[i]:<20} (stage {hook_targets[i]}): "
                                  f"{str(act.shape)} - {act.shape[1]} channels")
                        else:
                            print(f"  {model_names[i]:<20} (stage {hook_targets[i]}): {str(act.shape)}")
                    else:
                        print(f"  ✗ Failed to capture activation for {model_names[i]}")

                except Exception as e:
                    print(f"  ✗ Error with {model_names[i]}: {str(e)[:80]}")

            if len(activations) < 2:
                print("\n✗ Not enough activations captured. Skipping batch.")
                continue

            # Compare activations
            print(f"\nComputing {method.upper()} similarities:")
            print("-" * 70)

            comparison_count = 0
            for act1_i in range(len(activations)):
                for act2_i in range(act1_i + 1, len(activations)):
                    act1 = activations[act1_i]
                    act2 = activations[act2_i]

                    try:
                        sim = mean_correlation(batch_size, act1, act2)
                       
                        # Get the actual model names (not indices)
                        name1 = model_names[act1_i]
                        name2 = model_names[act2_i]
                        print(f"{name1:20s} vs {name2:20s}: {sim:.4f}")
                        comparison_count += 1
                    except Exception as e:
                        print(f"  ✗ Error comparing {model_names[act1_i]} vs {model_names[act2_i]}: {str(e)[:80]}")

            print(f"\n{comparison_count} comparisons completed successfully!")
            print("=" * 70)
            break  # Remove to process more batches
